# Vector hóa dữ liệu dạng bảng: mã hóa biến phân loại và chuẩn hóa

Mô hình học máy chỉ làm việc với số. Notebook này minh họa hai bước để biến một bảng dữ liệu thành các vector đặc trưng:

1. **Mã hóa one-hot** cho biến phân loại (giáo trình Chương 2).
2. **Chuẩn hóa** các đặc trưng số về cùng thang đo (giáo trình Chương 2).

## 1. Mã hóa one-hot

Biến phân loại có $K$ giá trị được thay bằng $K$ cột 0/1, mỗi mẫu có đúng một cột bằng 1. Không mã hóa thành 0, 1, 2 vì như vậy mô hình sẽ hiểu sai là có thứ tự và khoảng cách ("Chim" = 2 lớn gấp đôi "Chó" = 1).

In [1]:
import numpy as np
import pandas as pd

animals = pd.DataFrame({'animal': ['Mèo', 'Chó', 'Chim', 'Chó']})   # biến phân loại có 3 giá trị
pd.get_dummies(animals, columns=['animal'], dtype=int)              # mỗi giá trị thành một cột 0/1

,animal_Chim,animal_Chó,animal_Mèo
0,0,0,1
1,0,1,0
2,1,0,0
3,0,1,0


In [2]:
from sklearn.preprocessing import OneHotEncoder

houses = pd.DataFrame({'district': ['center', 'suburb', 'outskirts', 'suburb'],
                       'area': [60.0, 95.5, 150.0, 120.0]})
encoder = OneHotEncoder()                                      # bộ mã hóa one-hot
district_onehot = encoder.fit_transform(houses[['district']]).toarray()
print('Tên cột:', encoder.get_feature_names_out().tolist())
print(district_onehot)

Tên cột: ['district_center', 'district_outskirts', 'district_suburb']
[[1. 0. 0.]
 [0. 0. 1.]
 [0. 1. 0.]
 [0. 0. 1.]]


`OneHotEncoder` của scikit-learn **ghi nhớ** danh sách giá trị đã thấy khi `fit`, nhờ vậy tập kiểm tra và dữ liệu mới được mã hóa thành đúng các cột như tập huấn luyện.

## 2. Chuẩn hóa đặc trưng số

| Phương pháp | Công thức | Kết quả |
|---|---|---|
| Chuyển khoảng (min-max) | $x' = \dfrac{x - \min}{\max - \min}$ | Giá trị trong $[0, 1]$ |
| Chuẩn hóa theo phân phối chuẩn (z-score) | $x' = \dfrac{x - \bar{x}}{\sigma}$ | Trung bình 0, độ lệch chuẩn 1 |
| Chuẩn hóa theo norm | $\mathbf{x}' = \dfrac{\mathbf{x}}{\lVert \mathbf{x} \rVert_2}$ | Vector có độ dài 1 |

In [3]:
# Ví dụ trong giáo trình: diện tích trong khoảng [50, 200] m², nhà 120 m²
print('Min-max  :', round((120 - 50) / (200 - 50), 2))
# Tuổi trung bình 40, độ lệch chuẩn 10, người 50 tuổi
print('Z-score  :', (50 - 40) / 10)
# Vector (3, 4) có độ dài 5
x = np.array([3.0, 4.0])
print('Norm L2  :', x / np.linalg.norm(x))

Min-max  : 0.47
Z-score  : 1.0
Norm L2  : [0.6 0.8]


In [4]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

X = np.array([[60.0, 2, 3.0],       # diện tích, số phòng, khoảng cách
              [95.5, 3, 10.0],
              [150.0, 4, 20.0],
              [120.0, 3, 8.0]])
print('Min-max:\n', MinMaxScaler().fit_transform(X).round(3))
X_standard = StandardScaler().fit_transform(X)
print('Z-score:\n', X_standard.round(3))
print('Trung bình sau chuẩn hóa:', X_standard.mean(axis=0).round(3) + 0.0)   # + 0.0 để không in -0.
print('Độ lệch chuẩn sau chuẩn hóa:', X_standard.std(axis=0).round(3))

Min-max:
 [[0.    0.    0.   ]
 [0.394 0.5   0.412]
 [1.    1.    1.   ]
 [0.667 0.5   0.294]]
Z-score:
 [[-1.405 -1.414 -1.173]
 [-0.329  0.    -0.04 ]
 [ 1.322  1.414  1.578]
 [ 0.413  0.    -0.364]]
Trung bình sau chuẩn hóa: [0. 0. 0.]
Độ lệch chuẩn sau chuẩn hóa: [1. 1. 1.]


**Lưu ý quan trọng:** bộ chuẩn hóa chỉ được học (`fit`) trên **tập huấn luyện**, rồi dùng chính trung bình, độ lệch chuẩn đó để biến đổi tập kiểm tra (`transform`). Nếu `fit` trên toàn bộ dữ liệu, thông tin của tập kiểm tra bị "rò rỉ" vào quá trình huấn luyện.